# Module 2: First Principles of Vector Search

## What you will do

1. Turn text into a vector and look at what a collection needs to know about it.
2. Create your first Qdrant collection.
3. Upsert points with vectors and payloads.
4. Run a top-K query, then constrain it with a payload filter.
5. See why the payload index has to exist before you ingest.

**Tip:** every cell below is already run, so you can read it straight through. It uses Qdrant in local mode, so there is nothing to sign up for and no API key to paste.

Companion notebook to the [Module 2 lesson](https://qdrant.tech/course/beginners/module-2/).

## Setup

`qdrant-client[fastembed]` bundles local embedding models. Passing a `models.Document` lets the client embed text for us before upload and at query time, so we never handle raw vectors by hand. The model is `all-MiniLM-L6-v2`, the same one from Module 1.

In [ ]:
!pip install -q "qdrant-client[fastembed]" 

## 1. What a Collection Needs to Know

A collection is a container for points, and it is declared up front with two things it can never change silently: how many dimensions each vector has, and which distance metric compares them.

Those two numbers are not stylistic. Get the size wrong and every upsert fails. Get the metric wrong and your results are quietly worse rather than broken.

In [ ]:
from qdrant_client import QdrantClient, models

MODEL = "sentence-transformers/all-MiniLM-L6-v2"
VECTOR_SIZE = 384          # fixed by the model
DISTANCE = models.Distance.COSINE   # the default for text embeddings

# Local mode: an in-process Qdrant, ideal for notebooks and CI.
client = QdrantClient(":memory:")

# On Qdrant Cloud you would swap in:
# client = QdrantClient(url="https://YOUR-CLUSTER.cloud.qdrant.io:6333",
#                       api_key="YOUR_API_KEY")

client.create_collection(
    collection_name="articles",
    vectors_config=models.VectorParams(size=VECTOR_SIZE, distance=DISTANCE),
)

print(client.get_collection("articles").config.params.vectors)

size=384 distance=<Distance.COSINE: 'Cosine'> hnsw_config=None quantization_config=None on_disk=None datatype=None multivector_config=None


## 2. Index What You Will Filter, Before You Ingest

The collection is empty, and this is the moment to declare the payload fields you plan to filter on.

The ordering matters. Qdrant adds filter-aware edges to its vector index based on indexed payload values, and it can only add them for indexes that already exist when that index is built. Create a payload index after ingesting and you have to rebuild the vector index to get the benefit.

Local mode ignores payload indexes entirely, so the call below is a no-op here and will emit a warning. Write it anyway: it is the habit that transfers to a real server, and Module 4 goes into why.

In [ ]:
import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore")     # local mode warns that indexes do nothing here
    client.create_payload_index(
        collection_name="articles",
        field_name="category",
        field_schema=models.PayloadSchemaType.KEYWORD,
    )

print("payload index declared on 'category'")

payload index declared on 'category'


## 3. Points: Vector Plus Payload

A point is an id, one or more vectors, and a payload. The vector is what gets searched. The payload is everything you want back, or want to filter on, and it is ordinary JSON.

Note what goes in the payload below. `title` is there to be returned to the user. `category` is there to be filtered. `published` is there to be sorted or range-filtered later. None of them affect similarity.

In [ ]:
documents = [
    {"id": 1, "title": "Car repair guide",          "category": "automotive", "published": 2023},
    {"id": 2, "title": "Automobile maintenance 101", "category": "automotive", "published": 2024},
    {"id": 3, "title": "How to cook pasta",          "category": "food",       "published": 2024},
    {"id": 4, "title": "Best pizza in Chicago",      "category": "food",       "published": 2022},
]

client.upload_points(
    collection_name="articles",
    points=[
        models.PointStruct(
            id=doc["id"],
            vector=models.Document(text=doc["title"], model=MODEL),
            payload=doc,
        )
        for doc in documents
    ],
)

print("points in collection:", client.count("articles").count)

points in collection: 4


## 4. Your First Query

Same model at query time as at ingestion time. This is the rule that breaks the most beginner pipelines: two different models produce two different vector spaces, and nothing will warn you, you will just get nonsense rankings.

The query below shares no words with any stored title.

In [ ]:
results = client.query_points(
    collection_name="articles",
    query=models.Document(text="fixing my vehicle", model=MODEL),
    limit=4,
)

for r in results.points:
    print(f"{r.score:.3f}  {r.payload['title']:30} ({r.payload['category']})")

0.699  Car repair guide               (automotive)
0.565  Automobile maintenance 101     (automotive)
0.045  Best pizza in Chicago          (food)
0.032  How to cook pasta              (food)


Both automotive articles come back above both food articles, and the query contained neither the word "car" nor "automobile". That is the Module 1 embedding doing its job, now with storage and ranking around it.

Also worth noticing: every result carries its payload back. You did not have to look anything up in a second database.

## 5. Filtering

A filter is a hard constraint, not a ranking hint. It is evaluated while the search runs, so excluded points never occupy a slot in your top-K.

Run the same query twice, once unfiltered and once scoped to `food`, and watch the entire result set change rather than just reorder.

In [ ]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

def search(text, query_filter=None, limit=4):
    return client.query_points(
        collection_name="articles",
        query=models.Document(text=text, model=MODEL),
        query_filter=query_filter,
        limit=limit,
    ).points

food_only = Filter(must=[FieldCondition(key="category", match=MatchValue(value="food"))])

print("UNFILTERED 'fixing my vehicle':")
for r in search("fixing my vehicle"):
    print(f"   {r.score:.3f}  {r.payload['title']:30} ({r.payload['category']})")

print()
print("FILTERED to category=food:")
for r in search("fixing my vehicle", query_filter=food_only):
    print(f"   {r.score:.3f}  {r.payload['title']:30} ({r.payload['category']})")

UNFILTERED 'fixing my vehicle':
   0.699  Car repair guide               (automotive)
   0.565  Automobile maintenance 101     (automotive)
   0.045  Best pizza in Chicago          (food)
   0.032  How to cook pasta              (food)

FILTERED to category=food:
   0.045  Best pizza in Chicago          (food)
   0.032  How to cook pasta              (food)


The filtered run returns only food articles, and their scores are unchanged from the unfiltered run. The filter did not rescore anything. It removed candidates.

That distinction matters later: because filtering happens during retrieval rather than after it, a filtered search still returns a full top-K of valid results instead of a short list of leftovers.

## 6. Range Filters and Combining Conditions

`must` is AND, `should` is OR, and `must_not` excludes. Numeric and date fields take ranges. They compose in one filter object.

In [ ]:
from qdrant_client.models import Range

recent_automotive = Filter(
    must=[
        FieldCondition(key="category", match=MatchValue(value="automotive")),
        FieldCondition(key="published", range=Range(gte=2024)),
    ]
)

print("automotive AND published >= 2024:")
for r in search("fixing my vehicle", query_filter=recent_automotive):
    print(f"   {r.score:.3f}  {r.payload['title']:30} ({r.payload['published']})")

automotive AND published >= 2024:
   0.565  Automobile maintenance 101     (2024)


One result, because only one point satisfies both conditions. Note that `published` was never indexed, and in local mode that costs nothing. On a real server it would work but scan, and on Qdrant Cloud with strict mode on it would be rejected outright. Every field you filter on needs an index.

## 7. Similarity Under the Hood

Qdrant does not compare your query against every stored vector. It walks an HNSW graph, a layered structure where each layer is a sparser shortcut over the one below, so search starts coarse and refines.

Two consequences worth carrying forward.

It is **approximate**. HNSW trades a small amount of recall for a large amount of speed, so a top-K is very likely, not certainly, the true nearest neighbours.

It is **not always used**. Below a size threshold, scanning every vector is genuinely faster, so Qdrant does that instead. That is why a tiny notebook collection like this one returns exact results, and why timing measurements here tell you nothing about production.

In [ ]:
info = client.get_collection("articles")
print("points:            ", info.points_count)
print("vector size:       ", info.config.params.vectors.size)
print("distance:          ", info.config.params.vectors.distance)
print("hnsw m:            ", info.config.hnsw_config.m)
print("hnsw ef_construct: ", info.config.hnsw_config.ef_construct)
print("full_scan_threshold:", info.config.hnsw_config.full_scan_threshold, "(KB)")

points:             4
vector size:        384
distance:           Cosine
hnsw m:             16
hnsw ef_construct:  100
full_scan_threshold: 10000 (KB)


`m` is how many edges each node keeps, `ef_construct` is how hard the graph works while being built, and `full_scan_threshold` is the size below which Qdrant skips the graph. You will tune these in later work; for now the point is that they are collection-level settings you inherit by default.

## Your turn

Two changes to try.

Add a point in a third category and rerun the unfiltered query. Does it land where you expect relative to the existing four?

Then change `DISTANCE` to `models.Distance.EUCLID`, recreate the collection, and re-ingest. Compare the scores to the cosine run. They will not be on the same scale, which is exactly why the metric is declared once per collection rather than per query.

## What's next: Module 3

- Where dense-only search fails: exact codes, model numbers, and SKUs
- Sparse vectors, BM25, and the inverted index
- Hybrid search: running dense and sparse together and fusing the results

[Continue to Module 3](https://qdrant.tech/course/beginners/module-3/)